# 🥬 Vegetable Freshness Classification — Experiment Notebook

Notebook tổng hợp toàn bộ pipeline: **crawl → preprocess → train → evaluate → predict**.

> Đọc `CLAUDE.md` ở thư mục gốc để hiểu ngữ cảnh dự án trước khi chạy.

## 0. Thiết lập

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('CWD:', Path.cwd())

In [ ]:
import tensorflow as tf
print('TF :', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 1. Crawl ảnh (chạy 1 lần)

```bash
python crawler/crawl_images.py --target 10000
```

## 2. Tiền xử lý + split

```bash
python preprocessing/preprocess.py --src dataset/raw --dst dataset
```

In [ ]:
from preprocessing.augmentation import build_train_generator, build_eval_generator, visualize_augmentation
visualize_augmentation(Path('dataset/train'), Path('results/augmentation_demo.png'))

## 3. Huấn luyện

In [ ]:
# MobileNetV2 (nhẹ, nhanh)
!python models/train.py --model mobilenet --epochs 30 --ft-epochs 15

In [ ]:
# ResNet50 (mạnh hơn)
!python models/train.py --model resnet --epochs 30 --ft-epochs 15

## 4. Đánh giá

In [ ]:
!python evaluation/evaluate.py --model checkpoints/mobilenet_best.keras
!python evaluation/evaluate.py --model checkpoints/resnet_best.keras

In [ ]:
!python evaluation/plots.py

## 5. Dự đoán ảnh mới

In [ ]:
from app.predict import load_and_preprocess, predict_one
from tensorflow.keras.models import load_model
model = load_model('checkpoints/mobilenet_best.keras')
pred = predict_one(model, Path('dataset/test/fresh').glob('*').__next__(), ['fresh','rotten'], 224)
print(pred)